# LangChain Quick Start

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. `ChatOpenAI` 모델을 초기화하고 기본 호출(`invoke`)을 실행할 수 있어요
2. `HumanMessage`, `SystemMessage`를 활용해 역할 기반 대화를 구성할 수 있어요
3. 대화 히스토리를 수동으로 관리해 멀티턴 대화를 구현할 수 있어요
4. `stream`으로 실시간 응답을 받고, 다양한 모델 제공자를 전환하는 방법을 알 수 있어요

## 사전 지식

- Python 기초 문법 (리스트, 딕셔너리, for 반복문)
- `.env` 파일과 환경 변수 개념

## LangChain이란?

LangChain은 대형 언어 모델(LLM, Large Language Model) 기반 애플리케이션을 개발할 수 있도록 돕는 프레임워크(Framework)예요. 프롬프트(Prompt) 관리, 체인(Chain) 구성, 외부 도구 연동 등 LLM 애플리케이션 구축에 필요한 핵심 기능을 제공해요.

아래 다이어그램은 LangChain의 전체 아키텍처(Architecture)를 보여줘요. 사용자 입력이 어떤 경로로 최종 출력까지 이어지는지 확인해볼까요?

```mermaid
flowchart TD
    A[사용자 입력<br/>User Input]:::input --> B[프롬프트 템플릿<br/>PromptTemplate]:::process
    B --> C[LLM 모델<br/>ChatOpenAI]:::process
    C --> D{응답 방식}:::process
    D --> E[invoke<br/>단일 응답]:::output
    D --> F[stream<br/>실시간 스트리밍]:::output
    D --> G[batch<br/>일괄 처리]:::output
    E --> H[AIMessage<br/>응답 객체]:::output
    F --> H
    G --> H
    H --> I[OutputParser<br/>텍스트 추출]:::process
    I --> J[최종 결과<br/>문자열 또는 구조화 데이터]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef external fill:#d1ecf1,stroke:#17a2b8,color:#0c5460
```

In [1]:
from dotenv import load_dotenv

# .env 파일에서 OPENAI_API_KEY 등 환경 변수를 로드
load_dotenv()

True

## 1. 모델 초기화

`langchain_openai`의 `ChatOpenAI`를 사용하여 모델을 초기화해요. `ChatOpenAI`는 OpenAI의 채팅 완성(Chat Completion) API를 래핑한 클래스예요. 모델 이름을 `model` 파라미터로 지정하면 해당 모델이 연결돼요.

> **실무 팁**: `gpt-4o-mini`는 속도와 비용 측면에서 입문 실습에 적합해요. 복잡한 추론이 필요할 때는 `gpt-4o`로 교체해볼 수 있어요.

In [2]:
from langchain_openai import ChatOpenAI
import os

# ---------------------------------------------------
# ChatOpenAI 모델 초기화
# ---------------------------------------------------
# ChatOpenAI: OpenAI의 채팅 완성 API를 감싸는 LangChain 클래스
# model 파라미터로 사용할 모델 이름을 지정해요
# gpt-4o-mini는 속도·비용 면에서 입문 실습에 적합해요
model = ChatOpenAI(model="gpt-4o-mini")
model

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x137431150>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x1385d2dd0>, root_client=<openai.OpenAI object at 0x137430f10>, root_async_client=<openai.AsyncOpenAI object at 0x1385d2990>, model_name='gpt-4o-mini', model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)

## 2. 기본 사용법: invoke

가장 간단한 방법은 문자열을 직접 `invoke()`에 전달하는 거예요. LangChain이 내부적으로 문자열을 `HumanMessage(Human Message)` 객체로 자동 변환해요.

실행하면 `AIMessage(AI Message)` 객체가 반환돼요. 순수한 텍스트가 필요할 때는 `.content` 속성으로 추출해요.

In [3]:
# ---------------------------------------------------
# invoke()로 모델 호출하기
# ---------------------------------------------------
# 문자열을 직접 전달하면 LangChain이 자동으로 HumanMessage로 변환해요
# 반환값은 AIMessage 객체 — .content로 텍스트를 꺼낼 수 있어요
response = model.invoke("안녕하세요!")
response

AIMessage(content='안녕하세요! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 10, 'total_tokens': 20, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_668125ce6a', 'id': 'chatcmpl-DMO9Vg2wqBhjLijPzG2SvyoyZpmli', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019d1841-55a2-77a3-8b2e-eb1dc897f5a3-0', usage_metadata={'input_tokens': 10, 'output_tokens': 10, 'total_tokens': 20, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [4]:
# AIMessage 객체에서 텍스트만 추출
# .content: 모델이 생성한 응답 문자열을 담고 있어요
response.content

'안녕하세요! 어떻게 도와드릴까요?'

## 3. 메시지 객체 사용

앞서 문자열을 직접 전달했어요. 이제 메시지 객체를 사용하는 구조화된 방식을 살펴볼게요.

LangChain은 세 가지 핵심 메시지 타입을 제공해요:

| 클래스 | 역할 |
|--------|------|
| `SystemMessage` | AI의 페르소나, 규칙, 제약 조건을 정의해요 |
| `HumanMessage` | 사용자의 발화예요 |
| `AIMessage` | 모델의 응답이에요 (대화 히스토리에 저장할 때 사용해요) |

`SystemMessage`로 AI 역할을 먼저 정의하고, `HumanMessage`로 질문을 전달해볼게요.

In [5]:
# ---------------------------------------------------
# 메시지 객체로 역할 기반 대화 구성하기
# ---------------------------------------------------
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# 1단계 : 메시지 객체 생성
# SystemMessage : AI의 역할, 규칙, 페르소나 정의 ( 대화 시작 전 설정 )
system_msg = SystemMessage("너는 친근한 동생이야. 반말로 답변해줘.")

# HumanMessage : 사용자의 발화
human_msg = HumanMessage("뭐하니?")

# 2단계 : 리스트로 묶어서 invoke
# 순서가 매우 중요 : [SystemMessage, HumanMessage, AIMessage, HumanMessage, ...]
messages = [system_msg, human_msg]
response = model.invoke(messages)
response.content

'나는 너랑 대화하고 있어! 너는 뭐 하고 있어?'

## 4. 대화 히스토리 관리

LLM 자체는 상태를 저장하지 않아요. 이전 대화를 기억시키려면 이전 메시지들을 리스트에 누적해서 매번 전달해야 해요.

> **주의**: 대화가 길어질수록 매번 전달하는 토큰(Token) 수가 늘어나요. 토큰은 LLM이 텍스트를 처리하는 기본 단위로, 많을수록 비용이 증가해요. 실무에서는 오래된 메시지를 요약하거나 일부만 유지하는 방식으로 관리해요.

In [11]:
# ---------------------------------------------------
# 대화 히스토리를 리스트에 누적해 멀티턴 대화 구현하기
# ---------------------------------------------------
# ============================================================
# TODO: 아래 코드를 실행하고, AI가 "철수"를 기억하는지 확인해요
# 힌트: messages 리스트에 AIMessage와 HumanMessage를 번갈아 append()해요
# 예상 결과: 두 번째 응답에서 AI가 "철수"라는 이름을 언급해요
# ============================================================

# 1단계: 대화 시작
messages = [
    SystemMessage("너는 나의 충직한 부하야. 오버스러운 이모지와 함께 나를 모시는 것 처럼 답변해"),
    HumanMessage("내 이름은 철수야.")
]

# 2단계: 첫 번째 LLM 호출
response1 = model.invoke(messages) # response1 : AIMessage
print("첫 번째 응답:", response1.content)

# 3단계 : 이전 대화를 기억하기 위해 AI 응답과 새로운 질문을 리스트에 추가
# LLM은 상태(대화 내역)를 저장하지 않으므로, 전체 대화 맥락을 매번 전달해야 한다.

messages.append(AIMessage(content=response1.content)) # AI의 이전 답변만 보존
messages.append(HumanMessage("내 이름이 뭐였지?"))

response2 = model.invoke(messages)
print("두 번째 응답", response2.content)

첫 번째 응답: 안녕하세요, 철수님! 🤗✨ 당신의 충직한 부하로서 언제나 곁에서 도와드릴 준비가 되어 있습니다! 무슨 말씀이라도 해주세요! 🥰👑✨
두 번째 응답 철수님, 당신의 이름은 철수입니다! 😄✨ 맞죠? 언제나 당신을 기억하고 있습니다! 💖👑 다른 질문이 있으시면 말씀해 주세요! 🥳🌟


## 5. 파라미터 튜닝

대화 히스토리를 다뤄봤어요. 이제 모델 동작을 조정하는 파라미터를 살펴볼게요.

모델 초기화 시 주요 파라미터를 설정할 수 있어요:

| 파라미터 | 설명 |
|----------|------|
| `temperature` | 0에 가까울수록 일관된 답변, 1에 가까울수록 다양한 답변이 나와요 |
| `max_tokens` | 최대 출력 토큰 수예요. 응답이 잘리는 것을 방지하거나 비용을 제한할 때 사용해요 |

> **실무 팁**: 사실 확인이 중요한 업무(코드 생성, 요약)에는 `temperature=0`을 권장해요. 창의적인 글쓰기나 브레인스토밍에는 `0.7~1.0` 사이를 사용해볼 수 있어요.

In [20]:
# ---------------------------------------------------
# temperature·max_tokens 파라미터 조정하기
# ---------------------------------------------------
# ============================================================
# TODO: temperature 값을 0.0, 0.7, 1.5로 바꿔가며 응답 변화를 관찰해요
# 힌트: temperature가 높을수록 창의적(무작위)인 답변이 나와요
# 예상 결과: 낮은 값일수록 매번 비슷한 답변, 높을수록 매번 다른 답변
# ============================================================
model_with_params = ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0,
    max_tokens=100
)

response = model_with_params.invoke("파이썬에 대해 간단히 설명해줘")
print(response.content)

파이썬(Python)은 1991년 귀도 반 로섬(Guido van Rossum)에 의해 처음 개발된 고급 프로그래밍 언어입니다. 파이썬은 다음과 같은 특징을 가지고 있습니다:

1. **간결하고 읽기 쉬운 문법**: 파이썬은 코드가 직관적이고 가독성이 높아, 초보자도 쉽게 배울 수 있습니다.

2. **다양한 용도**


## 6. 스트리밍 응답

파라미터 튜닝을 마쳤어요. 이제 긴 응답을 더 나은 사용자 경험으로 전달하는 스트리밍(Streaming) 방식을 알아볼게요.

`invoke()`는 전체 응답 완료까지 대기하지만, `stream()`은 LLM이 토큰을 생성하는 즉시 하나씩 전달해요. 첫 토큰까지의 시간(TTFT, Time To First Token)이 줄어들어 사용자가 응답을 기다리는 체감 시간이 크게 단축돼요.

In [21]:
# ---------------------------------------------------
# stream()으로 토큰 단위 실시간 응답 받기
# ---------------------------------------------------
# invoke()는 전체 응답이 완성될 때까지 대기하지만
# stream()은 LLM이 토큰을 생성하는 즉시 chunk 단위로 전달해요
# → 사용자가 첫 글자를 보기까지의 시간(TTFT)이 크게 줄어들어요
for chunk in model.stream("파이썬에 대해서 간단히 정리해줘"):
    print(chunk.content, end="", flush=True)
    
print()

파이썬(Python)은 1991년 귀도 반 로썸(Guido van Rossum)에 의해 처음 개발된 고급 프로그래밍 언어입니다. 다양한 용도로 사용되며, 다음과 같은 주요 특징들이 있습니다.

1. **문법이 간결하고 직관적**: 파이썬은 읽기 쉽고 명확한 문법을 가지고 있어, 초보자도 쉽게 배울 수 있습니다.

2. **다양한 용도**: 웹 개발, 데이터 분석, 인공지능, 머신러닝, 자동화 스크립트, 게임 개발 등 다양한 분야에서 사용됩니다.

3. **풍부한 라이브러리**: 파이썬은 수많은 표준 라이브러리와 서드파티 라이브러리를 제공하여, 복잡한 작업도 간단하게 수행할 수 있게 도와줍니다. 예를 들어, 데이터 과학을 위한 `Pandas`, 시각화를 위한 `Matplotlib`, 머신러닝을 위한 `Scikit-learn` 등이 있습니다.

4. **크로스 플랫폼**: 파이썬은 Windows, macOS, Linux 등 다양한 운영체제에서 실행할 수 있습니다.

5. **역동적 타이핑**: 변수의 데이터 타입을 명시적으로 선언할 필요가 없으며, 실행 중에 타입이 결정됩니다.

6. **커뮤니티와 지원**: 파이썬은 활발한 커뮤니티와 방대한 양의 자료를 보유하고 있어 학습 및 문제 해결에 큰 도움이 됩니다.

파이썬은 현재 많은 기업과 기관에서 채택하고 있으며, 특히 데이터 분석과 인공지능 분야에서 그 중요성이 더욱 커지고 있습니다.


## 7. 딕셔너리 형식 메시지

앞서 메시지 객체를 사용했는데, 딕셔너리(dictionary) 형식으로도 동일하게 전달할 수 있어요. 특히 외부 API 응답이나 JSON 데이터를 그대로 활용할 때 유용해요.

In [22]:
# ---------------------------------------------------
# 딕셔너리(dict) 형식으로 메시지 전달하기
# ---------------------------------------------------
# {"role": "...", "content": "..."} 형태도 메시지 객체와 동일하게 동작해요
# 외부 API 응답이나 JSON 데이터를 그대로 활용할 때 편리해요
messages_dict = [
    {"role": "system", "content": "너는 요리 전문가야."},
    {"role": "user", "content": "김치찌개 만드는 법 알려줘"}
]

response = model.invoke(messages_dict)
print(response.content)

김치찌개는 한국의 대표적인 찌개 중 하나로, 간단하면서도 깊은 맛을 느낄 수 있는 요리입니다. 다음은 기본적인 김치찌개 레시피입니다.

### 재료
- 신김치 2컵 (익은 김치가 좋습니다.)
- 돼지고기 (목살 또는 삼겹살) 150g
- 두부 1/2모
- 대파 1대
- 양파 1개
- 마늘 3~4쪽
- 고춧가루 1~2큰술 (취향에 따라 조절)
- 국간장 1큰술
- 소금, 후추 약간
- 물 4컵
- 참기름 1큰술 (선택사항)

### 만들기
1. **재료 손질**: 
   - 신김치는 한 입 크기로 잘라주세요.
   - 돼지고기는 얇게 썰고, 양파와 대파는 채 썰고, 마늘은 다져주세요.
   - 두부도 먹기 좋은 크기로 자릅니다.

2. **팬에서 재료 볶기**: 
   - 냄비에 참기름을 두르고, 얇게 썬 돼지고기를 넣고 볶아주세요. 고기가 하얗게 변할 때까지 볶습니다.

3. **김치 추가**: 
   - 볶은 돼지고기에 잘라 놓은 김치를 넣고 5~10분 정도 볶아주세요. 김치가 물러지고 향이 올라올 때까지 볶는 것이 중요합니다.

4. **국물 끓이기**: 
   - 물 4컵을 넣고 끓입니다. 끓어오르면 다진 마늘과 국간장을 추가합니다.

5. **재료 넣기**: 
   - 끓기 시작하면 양파와 두부를 넣고 15분 정도 더 끓입니다. 그 사이에 나오는 거품은 걷어내주세요.

6. **마무리**: 
   - 대파를 넣고 소금, 후추로 간을 맞추고 마지막으로 한 번 더 끓입니다.

7. **서빙**: 
   - 완성된 김치찌개를 그릇에 담고, 밥과 함께 따뜻하게 서빙합니다.

이렇게 간단하게 김치찌개를 만들 수 있습니다. 취향에 따라 다양한 재료를 추가해보세요! 즐거운 요리 시간 되세요!


## 8. 다른 모델 제공자 사용

LangChain의 장점 중 하나는 모델 제공자(Provider)를 쉽게 교체할 수 있다는 거예요. Anthropic, Google 등 다양한 모델을 동일한 인터페이스로 사용할 수 있어요. 아래 코드 셀에 주석으로 예시를 확인할 수 있어요.

---

## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **`invoke()`**: 단일 입력을 처리하고 `AIMessage` 객체를 반환해요
- **메시지 타입**: `SystemMessage`(역할 정의), `HumanMessage`(사용자 입력), `AIMessage`(AI 응답)
- **대화 히스토리**: LLM은 상태를 저장하지 않으므로, 이전 메시지를 리스트에 누적해 매번 전달해야 해요
- **`stream()`**: 토큰 단위로 실시간 응답을 받아 사용자 체감 속도를 높여요
- **모델 교체**: 클래스 이름만 바꾸면 다른 제공자의 모델로 전환할 수 있어요

## 다음 노트북 예고

다음 `02_Chain.ipynb`에서는 `PromptTemplate`, `OutputParser`를 파이프(`|`) 연산자로 연결하는 **체이닝(Chaining)** 기법을 배워요. 지금까지 수동으로 조합했던 작업들을 선언적으로 표현하는 방법을 알아볼게요.

In [ ]:
# ---------------------------------------------------
# 다른 모델 제공자로 교체하기 (참고용 예시)
# ---------------------------------------------------
# LangChain의 강점: 클래스 이름만 바꾸면 다른 제공자의 모델로 전환 가능
# 나머지 invoke(), stream() 인터페이스는 동일하게 사용해요

# Anthropic Claude 사용 예시
# from langchain_anthropic import ChatAnthropic
# os.environ["ANTHROPIC_API_KEY"] = "sk-..."
# claude_model = ChatAnthropic(model="claude-sonnet-4-5")
# response = claude_model.invoke("안녕하세요!")
# print(response.content)

# Google Gemini 사용 예시
# from langchain_google_genai import ChatGoogleGenerativeAI
# os.environ["GOOGLE_API_KEY"] = "..."
# gemini_model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")
# response = gemini_model.invoke("안녕하세요!")
# print(response.content)